# Step by step testing

## 1) Core Implementation — Manual Prompt Engineering and Evaluation

### Loading models

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
MAX_PROMPT_TOKENS = 10
GENERATION_LENGTH = 32
TARGETS = ["grogu", "mando", "kuiil", "peli", "fennec"]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6339.43it/s]


In [ ]:
print(f"Vocab size: {tokenizer.vocab_size}")  # should be 50257

Vocab size: 50257


See how targets tokenize

In [ ]:
for t in TARGETS:
    ids = tokenizer.encode(t)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"  {t:10s} -> {ids} -> {pieces}")

  grogu      -> [70, 3828, 84] -> ['g', 'rog', 'u']
  mando      -> [22249, 78] -> ['mand', 'o']
  kuiil      -> [74, 9019, 346] -> ['k', 'ui', 'il']
  peli       -> [30242, 72] -> ['pel', 'i']
  fennec     -> [69, 29727, 66] -> ['f', 'enne', 'c']


### Main utility functions

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=GENERATION_LENGTH):
    """
    Given a text prompt, return GPT-2's continuation as a string.
    
    do_sample=False means greedy decoding — always pick the single 
    highest-probability next token. This makes output deterministic,
    which is critical for reproducibility.
    """
    # Convert prompt string -> tensor of token IDs
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    attention_mask = torch.ones_like(input_ids) # remove warning message
    
    with torch.no_grad():  # Don't track gradients, we're just generating
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic decoding
            pad_token_id=tokenizer.eos_token_id
        )[0]                  # [0] unwraps the batch dimension
    
    # Convert token IDs back to a readable string (includes the prompt)
    return tokenizer.decode(output_ids)[len(prompt):]

In [10]:
prompt = "The bounty hunter walked into"
output = generate(model, tokenizer, prompt)
print([prompt, output])

['The bounty hunter walked into', ' the office of the FBI\'s New York office and asked for a copy of the FBI\'s new report on the case.\n\n"I was told that the']


**Test Harness**  
Constraints: Prompts must (1) NOT contain the target name, (2) be ≤10 tokens when
tokenized, and (3) cause GPT-2 to generate the target in its output

In [18]:
def test_case(model, tokenizer, prompt, keyword, max_tokens=MAX_PROMPT_TOKENS):
    """
    Test whether a prompt satisfies all constraints AND generates the target.
    The prompt can be a raw string or a list of token IDs.
    
    Conditions checked:
      1. keyword NOT in prompt (case-insensitive) — no cheating
      2. prompt length <= max_tokens when re-tokenized from string form
      3. keyword appears somewhere in GPT-2's generated output
    
    Returns:
      (passed: bool, message: str)
    """
    # handle token ids and raw strings
    if type(prompt) == list: 
        prompt = tokenizer.decode(prompt)

    # Condition 1: keyword not in prompt
    if keyword.lower() in prompt.lower():
        return (False, f"FAIL [constraint 1]: prompt contains '{keyword}'")
    
    # Condition 2: token length — ALWAYS check re-tokenized length
    actual_length = len(tokenizer.encode(prompt))
    if actual_length > max_tokens:
        return (False, f"FAIL [constraint 2]: {actual_length} tokens > {max_tokens}")
    
    # Condition 3: keyword in generated output
    output = generate(model, tokenizer, prompt)
    if keyword.lower() in output.lower():
        return (True, f"PASS: output = {repr(output)}")
    else:
        return (False, f"FAIL [constraint 3]: output = {repr(output)}")

### Manual Prompt Engineering and Evaluation

In [ ]:
# Part A: Verify baselines fail as expected
print("=== Baseline prompts (should all fail) ===")
passed = False
baselines = ["Star Wars", "baby yoda", "bounty hunter", "din djarin"]
for prompt in baselines:
    for target in TARGETS:
        passed, msg = test_case(model, tokenizer, prompt, target)
        if passed:  # Flag any surprising passes
            print(f"  UNEXPECTED PASS: '{prompt}' -> {target}")
            passed = True

if not passed:
    print("All cases failed")

=== Baseline prompts (should all fail) ===
All cases failed


#### Substring/Prefix Inspired Prompts

The hypothesis here is that we can get GPT-2 to follow a pattern and produce the target word by using substrings of target characters as the prompt

In [ ]:
prompt = "ugu guu grog"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[0]) # target = grogu

['ugu guu grog', 'u grogu grogu grogu grogu grogu grogu grogu grogu grogu grogu g']


(True,
 "PASS: output = 'u grogu grogu grogu grogu grogu grogu grogu grogu grogu grogu g'")

In [ ]:
prompt = "odo guu mand"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[1]) # target = mando

['odo guu mand', 'o, yu mando, yu mando, yu mando, yu mando, yu mando, yu mando,']


(True,
 "PASS: output = 'o, yu mando, yu mando, yu mando, yu mando, yu mando, yu mando,'")

In [ ]:
prompt = "iil qiil kui"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[2]) # target = kuiil

['iil qiil kui', 'il qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil q']


(True,
 "PASS: output = 'il qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil q'")

In [ ]:
prompt = "i u li gr eli pel"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[3]) # target = peli

['i u li gr eli pel', 'i u li gr eli peli u li gr eli peli u li gr eli peli u li gr eli peli u li gr']


(True,
 "PASS: output = 'i u li gr eli peli u li gr eli peli u li gr eli peli u li gr eli peli u li gr'")

In [ ]:
prompt = "nnec fnec-fx ennec fen"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[4]) # target = fennec

['nnec fnec-fx ennec fen', 'nec fennec fennec fennec fennec fennec fennec fennec fennec fennec fennec f']


(True,
 "PASS: output = 'nec fennec fennec fennec fennec fennec fennec fennec fennec fennec fennec f'")

Initially, the raw substrings alone were not enough to produce the target results. For example, the prompt 'gr ogu' produced the output of 'o, oguo oguo, oguo oguo, oguo oguo, oguo oguo, ogu'. Now this output is somewhat nonsensical, however it is clear to see that there is a pattern of the last characters of the prompt being repeated multiple times. Through manual fiddling, it was found that GPT-2 paid a lot of attention to the tokens on the ends of the prompt and would often repeat those characters in a nonsensical prompt. Thus is made it possible to use substrings of characters of the target word on the ends of the prompt in order to force GPT-2 to recreate the target word. There was interesting behavior for the importance of tokens between the ends of the prompt as well. For the most part, the middle tokens served to force GPT-2 to interpret the prompt as nonsensical and thus be prone to repeating characters of the prompt (sometimes without middle tokens, the output would be real human language). The most significant role the middle tokens played however, was to tune how and where characters appeared in the output. Using these principles and a lot of trial and error, prompts were manually fine tuned to reach all target words showing that this strategy is valid.

### ======== TODO ===========
Add 4 more manual prompting strategies similar to that of above. You can add a new header/section by adding a markdown cell and adding `#### Section Name` and then putting your code there.  (P.S. a newline in markdown is two spaces at the end of the line)  
Note that, "For EACH approach: document the exact prompt, show GPT-2’s output, explain your hypothesis, and analyze why it worked/failed."

## 3) Automated Prompt Search 